# CEG-WM Stage A2 HF Colab execution

This notebook runs the exact reviewed revision and exports incomplete HF-anchor evidence only. It does not evaluate the three preregistered attacks or LPIPS, and it cannot decide HF or Stage A. Authenticate to Hugging Face in this Colab session when prompted; no token is embedded. If the GitHub repository becomes private, authenticate Git in Colab before the exact fetch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import re, uuid
RUN_MODE = input('Run mode [fresh/resume]: ').strip().lower()
CHECKPOINT_INTERVAL_HOURS = float(input('Checkpoint interval hours [1.0-2.0, default 2.0]: ') or '2.0')
if not 1.0 <= CHECKPOINT_INTERVAL_HOURS <= 2.0:
    raise ValueError('checkpoint interval must be in [1.0, 2.0]')
if RUN_MODE == 'fresh':
    fresh_run_id = 'a2hf-' + uuid.uuid4().hex[:16]
elif RUN_MODE == 'resume':
    fresh_run_id = input('Existing run id: ').strip()
else:
    raise ValueError('run mode must be fresh or resume')
if re.fullmatch(r'[a-z0-9][a-z0-9-]{7,63}', fresh_run_id) is None:
    raise ValueError('invalid run id')
drive_relative_dir = f'CEG-WM/stage_a2_hf/{fresh_run_id}'
drive_run_dir = Path('/content/drive/MyDrive') / drive_relative_dir
if RUN_MODE == 'fresh':
    if drive_run_dir.exists():
        raise RuntimeError('fresh Drive run directory already exists')
    drive_run_dir.mkdir(parents=True, exist_ok=False)
    resume_zip_path = resume_checksum_path = None
else:
    if not drive_run_dir.is_dir():
        raise RuntimeError('resume Drive run directory does not exist')
    resume_zip_name = input('Checkpoint ZIP filename: ').strip()
    resume_checksum_name = input('Checkpoint checksum filename: ').strip()
    if Path(resume_zip_name).name != resume_zip_name or Path(resume_checksum_name).name != resume_checksum_name:
        raise ValueError('checkpoint inputs must be filenames in this run directory')
    resume_zip_path = drive_run_dir / resume_zip_name
    resume_checksum_path = drive_run_dir / resume_checksum_name
    if not (resume_zip_path.is_file() and resume_checksum_path.is_file()):
        raise RuntimeError('explicit resume checkpoint pair is missing')
if any(path.suffix not in {'.zip', '.sha256'} for path in drive_run_dir.iterdir()):
    raise RuntimeError('Drive run directory may contain only ZIP/checksum files')
print({'run_id': fresh_run_id, 'mode': RUN_MODE, 'checkpoint_interval_hours': CHECKPOINT_INTERVAL_HOURS, 'drive_relative_dir': drive_relative_dir})


In [ ]:
import getpass, os, re, shutil, subprocess, sys
APPROVED_EXECUTION_EXACT = input('Approved 40-hex execution exact: ').strip().lower()
MODEL_REVISION = input('Frozen 40-hex SD3.5 model revision: ').strip().lower()
if re.fullmatch(r'[0-9a-f]{40}', APPROVED_EXECUTION_EXACT) is None:
    raise ValueError('invalid approved execution exact')
if re.fullmatch(r'[0-9a-f]{40}', MODEL_REVISION) is None:
    raise ValueError('invalid model revision')
repo = Path('/content/CEG-WM-exact')
if repo.exists():
    raise RuntimeError('fresh checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', APPROVED_EXECUTION_EXACT], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if resolved_exact != APPROVED_EXECUTION_EXACT:
    raise RuntimeError('resolved exact mismatch')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)
hf_token = getpass.getpass('Hugging Face token (not stored): ')
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
del hf_token


In [ ]:
detection_key = getpass.getpass('Stage-A detection key (not stored): ')
local_output_root = Path('/content') / (fresh_run_id + '-local-results')
runner_env = dict(os.environ)
runner_env['CEGWM_STAGE_A_DETECTION_KEY'] = detection_key
command = [sys.executable, '-m', 'experiments.stage_a.run_hf_a2_colab', '--repo-root', str(repo), '--output-root', str(local_output_root), '--expected-exact', APPROVED_EXECUTION_EXACT, '--model-revision', MODEL_REVISION, '--run-id', fresh_run_id, '--checkpoint-sink', str(drive_run_dir), '--checkpoint-interval-hours', str(CHECKPOINT_INTERVAL_HOURS)]
if RUN_MODE == 'resume':
    command.extend(['--resume-zip', str(resume_zip_path), '--resume-checksum', str(resume_checksum_path)])
process = subprocess.Popen(command, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
for line in process.stdout:
    if line.startswith(('CEGWM_PROGRESS ', 'CEGWM_SUMMARY ', 'CEGWM_FATAL ')):
        print(line.rstrip())
runner_rc = process.wait()
runner_env.pop('CEGWM_STAGE_A_DETECTION_KEY', None)
detection_key = ''
del detection_key, runner_env
print({'run_id': fresh_run_id, 'runner_rc': runner_rc})


In [ ]:
import hashlib, json
local_run_dir = local_output_root / fresh_run_id
receipt_path = local_run_dir / 'receipt.json'
result_path = local_run_dir / 'result.json'
zip_path = local_run_dir / f'{fresh_run_id}.zip'
if not (receipt_path.is_file() and result_path.is_file() and zip_path.is_file()):
    raise RuntimeError('runner did not produce the safe result package')
receipt = json.loads(receipt_path.read_text())
result = json.loads(result_path.read_text())
if receipt['run_id'] != fresh_run_id or result['run_id'] != fresh_run_id:
    raise RuntimeError('run identity mismatch')
if receipt['resolved_exact'] != APPROVED_EXECUTION_EXACT or result['resolved_exact'] != APPROVED_EXECUTION_EXACT:
    raise RuntimeError('result exact mismatch')
if receipt['rc'] != runner_rc or result['rc'] != runner_rc:
    raise RuntimeError('runner/result RC mismatch')
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
checksum_path = local_run_dir / f'{fresh_run_id}.zip.sha256'
checksum_path.write_text(zip_sha256 + '  ' + zip_path.name + '\n')
drive_zip = drive_run_dir / zip_path.name
drive_checksum = drive_run_dir / checksum_path.name
if drive_zip.exists() or drive_checksum.exists():
    raise RuntimeError('final Drive artifacts refuse overwrite')
shutil.copy2(zip_path, drive_zip)
shutil.copy2(checksum_path, drive_checksum)
if hashlib.sha256(drive_zip.read_bytes()).hexdigest() != zip_sha256:
    raise RuntimeError('Drive ZIP hash mismatch')
summary = {'run_id': fresh_run_id, 'resolved_exact': APPROVED_EXECUTION_EXACT, 'status': receipt['status'], 'rc': runner_rc, 'drive_relative_dir': drive_relative_dir, 'zip_sha256': zip_sha256}
print(summary)
if runner_rc != 0:
    raise RuntimeError('runner completed with retained operational failures')
